# Quickstart: U.S. Natality Harmonization Dataset

This notebook demonstrates how to load and explore the harmonized natality dataset. It covers:

1. Loading the Parquet files
2. Basic filters and summary statistics
3. Trend analysis (LBW rate, preterm rate, cesarean rate)
4. Subgroup comparisons by race/ethnicity
5. Working with the linked birth-infant death file

**Requirements:** `pip install pyarrow pandas matplotlib`

**Data files:** Download from [Zenodo (DOI forthcoming)] and place in `output/harmonized/` and `output/convenience/`, or adjust the paths below.

In [ ]:
import pyarrow.parquet as pq
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Adjust this if your data lives elsewhere
DATA_DIR = Path("../output")

V2_DERIVED = DATA_DIR / "harmonized" / "natality_v2_harmonized_derived.parquet"
V2_RESIDENTS = DATA_DIR / "convenience" / "natality_v2_residents_only.parquet"
V3_DERIVED = DATA_DIR / "harmonized" / "natality_v3_linked_harmonized_derived.parquet"
V3_RESIDENTS = DATA_DIR / "convenience" / "natality_v3_linked_residents_only.parquet"

## 1. Inspect the schema

The full file is ~139M rows. Start by reading just the metadata to see what columns are available.

In [ ]:
schema = pq.read_schema(V2_DERIVED)
print(f"Columns: {len(schema.names)}\n")
for i, name in enumerate(schema.names):
    print(f"  {i+1:2d}. {name:40s} {schema.field(name).type}")

## 2. Load a single year

For quick exploration, read only the columns and rows you need. Parquet makes this efficient.

In [ ]:
# Read 2023 resident births with a few key columns
cols = [
    "year", "maternal_age", "maternal_race_ethnicity_5",
    "birthweight_grams_clean", "low_birthweight", "preterm_lt37",
    "gestational_age_weeks_clean", "singleton", "infant_sex",
]

df_2023 = (
    pq.read_table(V2_RESIDENTS, columns=cols, filters=[("year", "=", 2023)])
    .to_pandas()
)

print(f"Rows: {len(df_2023):,}")
df_2023.head()

In [ ]:
# Quick summary stats for 2023
print("=== 2023 Resident Births ===\n")
print(f"Total births:       {len(df_2023):>12,}")
print(f"Mean birthweight:   {df_2023['birthweight_grams_clean'].mean():>12,.0f} g")
print(f"Mean maternal age:  {df_2023['maternal_age'].mean():>12.1f}")
print(f"LBW rate:           {df_2023['low_birthweight'].mean():>12.2%}")
print(f"Preterm rate:       {df_2023['preterm_lt37'].mean():>12.2%}")
print(f"Singleton rate:     {df_2023['singleton'].mean():>12.2%}")
print(f"Male %:             {(df_2023['infant_sex'] == 'M').mean():>12.2%}")
print(f"\nRace/ethnicity distribution:")
print(df_2023["maternal_race_ethnicity_5"].value_counts(normalize=True).round(4))

## 3. Trend analysis with streaming (no full-file load)

For 35-year trends you don't need to load the entire file into memory. Use `iter_batches` to stream through the data and accumulate per-year statistics.

In [ ]:
from collections import defaultdict

# Stream through the residents-only file, accumulating counts by year
cols = ["year", "low_birthweight", "preterm_lt37", "delivery_method_recode"]
accum = defaultdict(lambda: {"births": 0, "lbw": 0, "lbw_known": 0,
                              "preterm": 0, "preterm_known": 0,
                              "cesarean": 0, "deliv_known": 0})

pf = pq.ParquetFile(V2_RESIDENTS)
for batch in pf.iter_batches(batch_size=500_000, columns=cols):
    df = batch.to_pandas()
    for year, g in df.groupby("year"):
        a = accum[year]
        a["births"] += len(g)
        lbw = g["low_birthweight"].dropna()
        a["lbw"] += lbw.sum()
        a["lbw_known"] += len(lbw)
        pt = g["preterm_lt37"].dropna()
        a["preterm"] += pt.sum()
        a["preterm_known"] += len(pt)
        # Cesarean: codes 3+4 for 1990-2004 (DELMETH5), code 2 for 2005+
        dm = g["delivery_method_recode"].dropna()
        if year <= 2004:
            a["cesarean"] += dm.isin([3, 4]).sum()
            a["deliv_known"] += dm.isin([1, 2, 3, 4]).sum()
        else:
            a["cesarean"] += (dm == 2).sum()
            a["deliv_known"] += dm.isin([1, 2]).sum()

trends = pd.DataFrame([
    {"year": y,
     "births": a["births"],
     "lbw_pct": 100 * a["lbw"] / a["lbw_known"] if a["lbw_known"] else None,
     "preterm_pct": 100 * a["preterm"] / a["preterm_known"] if a["preterm_known"] else None,
     "cesarean_pct": 100 * a["cesarean"] / a["deliv_known"] if a["deliv_known"] else None}
    for y, a in sorted(accum.items())
])

print(f"Streamed {trends['births'].sum():,} resident births across {len(trends)} years")
trends.head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].plot(trends["year"], trends["lbw_pct"], "o-", markersize=3, color="#2171b5")
axes[0].set_title("Low Birthweight Rate (%)")
axes[0].set_ylabel("%")
axes[0].set_xlabel("Year")
axes[0].set_ylim(6, 9.5)

axes[1].plot(trends["year"], trends["preterm_pct"], "o-", markersize=3, color="#cb181d")
axes[1].set_title("Preterm Birth Rate (%)")
axes[1].set_ylabel("%")
axes[1].set_xlabel("Year")
axes[1].axvline(2003, color="gray", ls="--", lw=0.8, label="Certificate revision")
axes[1].axvline(2014, color="gray", ls=":", lw=0.8, label="OE gestation")
axes[1].legend(fontsize=8)

axes[2].plot(trends["year"], trends["cesarean_pct"], "o-", markersize=3, color="#238b45")
axes[2].set_title("Cesarean Delivery Rate (%)")
axes[2].set_ylabel("%")
axes[2].set_xlabel("Year")

for ax in axes:
    ax.grid(True, alpha=0.3)

fig.suptitle("U.S. Birth Outcome Trends, 1990-2024 (Resident Births)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. Subgroup analysis: LBW rate by race/ethnicity

Demonstrates a common research use case: stratifying outcomes by maternal race/ethnicity over time.

In [ ]:
# Stream LBW by year and race/ethnicity
cols = ["year", "maternal_race_ethnicity_5", "low_birthweight"]
race_accum = defaultdict(lambda: defaultdict(lambda: {"lbw": 0, "known": 0}))

pf = pq.ParquetFile(V2_RESIDENTS)
for batch in pf.iter_batches(batch_size=500_000, columns=cols):
    df = batch.to_pandas().dropna(subset=["maternal_race_ethnicity_5", "low_birthweight"])
    for (year, race), g in df.groupby(["year", "maternal_race_ethnicity_5"]):
        race_accum[year][race]["lbw"] += g["low_birthweight"].sum()
        race_accum[year][race]["known"] += len(g)

race_trends = pd.DataFrame([
    {"year": y, "race_ethnicity": r,
     "lbw_pct": 100 * v["lbw"] / v["known"]}
    for y, races in sorted(race_accum.items())
    for r, v in races.items()
])

race_pivot = race_trends.pivot(index="year", columns="race_ethnicity", values="lbw_pct")
race_pivot.tail()

In [ ]:
COLORS = {
    "NH_white": "#4292c6",
    "NH_black": "#e6550d",
    "Hispanic": "#41ab5d",
    "NH_aian": "#9e9ac8",
    "NH_asian_pi": "#f768a1",
}

fig, ax = plt.subplots(figsize=(10, 5))
for race in ["NH_white", "NH_black", "Hispanic", "NH_asian_pi", "NH_aian"]:
    if race in race_pivot.columns:
        ax.plot(race_pivot.index, race_pivot[race], "-", lw=1.8,
                color=COLORS.get(race, "gray"), label=race)

ax.set_xlabel("Year")
ax.set_ylabel("Low Birthweight Rate (%)")
ax.set_title("LBW Rate by Maternal Race/Ethnicity, 1990-2024")
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Working with the linked birth-infant death file (V3)

The V3 dataset links each birth to its infant death record (if any), enabling infant mortality analysis.

In [ ]:
# Compute annual infant mortality rate (per 1,000 live births), residents only
cols = ["year", "infant_death", "neonatal_death", "postneonatal_death"]
imr_accum = defaultdict(lambda: {"births": 0, "deaths": 0, "neo": 0, "post": 0})

pf = pq.ParquetFile(V3_RESIDENTS)
for batch in pf.iter_batches(batch_size=500_000, columns=cols):
    df = batch.to_pandas()
    for year, g in df.groupby("year"):
        a = imr_accum[year]
        a["births"] += len(g)
        a["deaths"] += g["infant_death"].sum()
        a["neo"] += g["neonatal_death"].sum()
        a["post"] += g["postneonatal_death"].sum()

imr = pd.DataFrame([
    {"year": y,
     "births": a["births"],
     "deaths": a["deaths"],
     "imr": 1000 * a["deaths"] / a["births"],
     "neonatal_imr": 1000 * a["neo"] / a["births"],
     "postneonatal_imr": 1000 * a["post"] / a["births"]}
    for y, a in sorted(imr_accum.items())
])

print("Infant Mortality Rate per 1,000 live births (residents)")
imr[["year", "births", "deaths", "imr", "neonatal_imr", "postneonatal_imr"]].to_string(index=False)
imr

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(imr["year"], imr["imr"], "o-", lw=2, color="#252525", label="Total IMR", markersize=4)
ax.plot(imr["year"], imr["neonatal_imr"], "s--", lw=1.5, color="#2171b5", label="Neonatal", markersize=3)
ax.plot(imr["year"], imr["postneonatal_imr"], "^--", lw=1.5, color="#cb181d", label="Postneonatal", markersize=3)

ax.set_xlabel("Year")
ax.set_ylabel("IMR (per 1,000 live births)")
ax.set_title("U.S. Infant Mortality Rate, 2005-2023 (Resident Births)")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Cause-of-death distribution (V3 linked)

The `cause_group` column classifies infant deaths into 13 standard NCHS categories.

In [ ]:
# Read cause of death for the most recent year (2023)
deaths_2023 = (
    pq.read_table(
        V3_RESIDENTS,
        columns=["year", "infant_death", "cause_group"],
        filters=[("year", "=", 2023), ("infant_death", "=", True)],
    )
    .to_pandas()
)

cause_dist = (
    deaths_2023["cause_group"]
    .value_counts()
    .reset_index()
    .rename(columns={"count": "deaths"})
)
cause_dist["pct"] = (100 * cause_dist["deaths"] / cause_dist["deaths"].sum()).round(1)
print(f"2023 infant deaths: {len(deaths_2023):,}\n")
cause_dist

## Tips

- **Memory**: The full V2 derived file is ~139M rows. Use `iter_batches()` for streaming, or `filters=` to read specific years. The residents-only convenience files are slightly smaller.
- **Comparability**: Check `docs/COMPARABILITY.md` before comparing across eras. Key breaks: gestation measurement (2003, 2014), education/smoking (2003 revision), marital status (2017 California break), race bridging (pre-2003 approximate, 2020+ reconstructed from MRACE6).
- **Marital status 2017+**: ~11-12% null from California. Use `marital_reporting_flag == True` (2014+) to restrict to reporting states.
- **Race/ethnicity 2020+**: `maternal_race_ethnicity_5` is reconstructed from MRACE6 detail codes; multiracial births (~3%) are null. Use `race_bridge_method` to identify the derivation era.
- **Diabetes / HTN**: Prefer `diabetes_any_bool`, `hypertension_chronic_bool`, `hypertension_gestational_bool` over the raw integer columns — the raw fields use 9=unknown which passes `IS NOT NULL` filters.
- **Cesarean rates**: Use the era-aware crosswalk shown above (DELMETH5 codes 3+4 for 1990-2004, DMETH_REC code 2 for 2005+).
- **Linked file**: Use unweighted counts for cohort analyses per NCHS guidance. The `record_weight` column exists but is not recommended for cohort-based IMR.
- **Missingness check**: Before any trend analysis, inspect `output/validation/harmonized_missingness_by_year.csv` for null-rate breaks, or run `python scripts/05_validate/harmonized_missingness.py` to regenerate.
- **Parquet versioning**: Convenience files embed `pipeline_git_hash` and `pipeline_build_timestamp` in Parquet metadata. Check `output/convenience/PROVENANCE.md` for SHA-256 checksums.
- **Full schema**: See `metadata/harmonized_schema.csv` for every column's provenance, allowed values, and comparability class.